In [1]:
#loading filtered rna and atac data
import anndata
from anndata import AnnData
import pandas as pd

pb_atac_ct_time = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/ATAC/celltypes_times/agg_atac_ct_time.h5ad")
pb_rna_ct_time = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/RNA/celltypes_times/agg_rna_ct_time.h5ad") 

window_sizes = ["5kb", "10kb", "20kb", "30kb", "40kb", "50kb", "60kb", "70kb", "80kb", "90kb", "100kb"]
base_path = "/home/fgsasse_lrs_1/Downloads/BA/BA_data/TSS windows/20 windows"

gene_peaks_neg = {}
for window in window_sizes:
    gene_peaks_neg[window] = pd.read_csv(f"{base_path}/gene_peak_assignments_{window}_neg_upst.csv")

gene_peaks_pos = {}
for window in window_sizes:
    gene_peaks_pos[window] = pd.read_csv(f"{base_path}/gene_peak_assignments_{window}_pos_downst.csv")

combined_gene_peaks = {}
for window in window_sizes:
    combined_gene_peaks[window] = pd.read_csv(f"{base_path}/combined_gene_peak_assignments_{window}.csv")

In [2]:
#check dimensions of the data
print(pb_atac_ct_time.X.shape)
print(pb_rna_ct_time.X.shape)

print(pb_atac_ct_time.obs_names)
print(pb_rna_ct_time.obs_names)

(196, 640834)
(196, 32057)
Index(['NMPs__0somites', 'NMPs__5somites', 'NMPs__10somites',
       'NMPs__15somites', 'PSM__0somites', 'PSM__5somites', 'PSM__10somites',
       'PSM__15somites', 'PSM__20somites', 'differentiating_neurons__5somites',
       ...
       'spinal_cord__5somites', 'spinal_cord__10somites',
       'spinal_cord__15somites', 'spinal_cord__20somites',
       'spinal_cord__30somites', 'tail_bud__0somites', 'tail_bud__5somites',
       'tail_bud__10somites', 'tail_bud__15somites', 'tail_bud__20somites'],
      dtype='object', length=196)
Index(['epidermis__15somites', 'pronephros__15somites', 'hindbrain__15somites',
       'spinal_cord__15somites', 'neural_optic2__15somites',
       'neural_floor_plate__15somites', 'neural_crest2__15somites',
       'PSM__15somites', 'optic_cup__15somites',
       'lateral_plate_mesoderm__15somites',
       ...
       'neural_crest2__10somites', 'muscle__10somites',
       'epidermis2__10somites', 'floor_plate__10somites',
       'he

In [3]:
#Reindex the dataframes to ensure they are in the same order
pb_atac_ct_time = pb_atac_ct_time[pb_rna_ct_time.obs_names, :]
pb_rna_ct_time = pb_rna_ct_time[pb_atac_ct_time.obs_names, :]
print(pb_atac_ct_time.obs_names.equals(pb_rna_ct_time.obs_names))  # Should return True

pb_rna_ct_time.shape

True


(196, 32057)

In [4]:
combined_gene_peaks["100kb"]["gene_id"].nunique()

19380

In [5]:
combined_gene_peaks["100kb"].head()

,gene_id,assigned_peaks
0,a1cf,"['12-6186644-6187111', '12-6190483-6191143', '..."
1,a2ml,"['15-21143884-21144162', '15-21033797-21034001..."
2,aaas,"['9-323370-323691', '9-209853-211311', '9-2304..."
3,aacs,"['5-18949828-18950713', '5-18811696-18812645',..."
4,aadac,"['15-1052335-1053001', '15-1029184-1030074', '..."


In [6]:
print(pb_atac_ct_time.X)
print(pb_rna_ct_time.X)

[[4.22694502e-07 6.76311203e-07 1.94439471e-06 ... 5.91772302e-07
  2.24822752e-03 1.84066548e-03]
 [0.00000000e+00 7.52470643e-07 9.40588304e-07 ... 9.40588304e-07
  2.35843111e-03 1.90958238e-03]
 [1.83118590e-07 1.37338943e-06 9.15592951e-07 ... 1.64806731e-06
  2.20300820e-03 1.78027893e-03]
 ...
 [4.10770344e-07 1.78000482e-06 5.06616757e-06 ... 1.09538758e-06
  2.15722892e-03 1.92391137e-03]
 [4.57123736e-07 6.85685604e-07 1.59993308e-06 ... 4.57123736e-07
  2.21110751e-03 2.02185829e-03]
 [2.63333289e-07 1.84333302e-06 3.68666604e-06 ... 2.10666631e-06
  1.79487969e-03 1.63345639e-03]]
[[1.70213019e-06 6.38298823e-07 1.48936392e-06 ... 3.72681406e-03
  4.25532549e-07 0.00000000e+00]
 [1.87578197e-06 4.68945492e-07 9.37890983e-07 ... 3.90068860e-03
  1.40683647e-06 4.68945492e-07]
 [8.51931471e-07 1.13590863e-06 3.12374873e-06 ... 4.00947348e-03
  8.51931471e-07 1.41988578e-06]
 ...
 [0.00000000e+00 0.00000000e+00 7.77981868e-07 ... 4.67333708e-03
  0.00000000e+00 0.00000000e+00]

In [7]:
#taking the minimum non-zero value in the pb_rna_ct_time matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np
non_zero_mask = (pb_rna_ct_time.X > 0)
epsilon_rna = np.min(pb_rna_ct_time.X[non_zero_mask])

#taking the minimum non-zero value in the pb_atac_ct_time matrix to add it to all values before log transformation to avoid taking log of zero
non_zero_mask = (pb_atac_ct_time.X > 0)
epsilon_at= np.min(pb_atac_ct_time.X[non_zero_mask])

In [8]:
#log transformation (for norm. distribution) & scaling 10000000+1
#log scaling both datasets by multiplying by 10 million and adding 1 to avoid log(0) issues, then taking log10
pb_rna_ct_time.X = np.log10((pb_rna_ct_time.X+ epsilon_rna)*10000000)
pb_atac_ct_time.X = np.log10((pb_atac_ct_time.X+ epsilon_at)*10000000)
print(pb_atac_ct_time.X)
print(pb_rna_ct_time.X)

/tmp/ipykernel_3215882/1176312871.py:3: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  pb_rna_ct_time.X = np.log10((pb_rna_ct_time.X+ epsilon_rna)*10000000)


[[ 0.66042139  0.85196037  1.29649673 ...  0.79699712  4.35184699
   4.26498309]
 [-0.4579541   0.8961447   0.9891942  ...  0.9891942   4.37262961
   4.28094632]
 [ 0.33836884  1.14867262  0.9779203  ...  1.22605953  4.34302298
   4.25049655]
 ...
 [ 0.64895274  1.25883888  1.70765575 ...  1.05316491  4.33390325
   4.28419292]
 [ 0.69193081  0.85764786  1.21345679 ...  0.69193081  4.3446167
   4.3057582 ]
 [ 0.47446497  1.273735    1.57071843 ...  1.33071883  4.25404377
   4.21311681]]
[[1.25848578 0.87479245 1.20428309 ... 4.57135069 0.72978795 0.04624963]
 [1.29820203 0.76356469 1.02082851 ... 4.59115367 1.18129282 0.76356469]
 [0.9837023  1.0959172  1.50987238 ... 4.60309939 0.9837023  1.18501005]
 ...
 [0.04624963 0.04624963 0.94900871 ... 4.66963744 0.04624963 0.04624963]
 [1.15741357 0.04624963 1.44129868 ... 4.6874499  0.04624963 1.44129868]
 [0.04624963 1.46040775 1.63088453 ... 4.6812724  0.04624963 0.04624963]]


/tmp/ipykernel_3215882/1176312871.py:4: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  pb_atac_ct_time.X = np.log10((pb_atac_ct_time.X+ epsilon_at)*10000000)


In [ ]:
#taking the minimum non-zero value in the pb_rna_ct_time matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np

non_zero_mask = (pb_rna_ct_time.X > 0)
epsilon_rna = np.min(pb_rna_ct_time.X[non_zero_mask])


In [ ]:
#Plotting the distribution of the value of the genes across all cell types in a boxplot for RNA data
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

P_rna_scaled = np.log10((pb_rna_ct_time.X+ epsilon_rna)*10000000) 

# RNA counts x samples
P_rna_scaled = P_rna_scaled.T.astype(float)

# convert to long format for seaborn
P_rna_scaled_plot = pd.DataFrame(P_rna_scaled, columns=pb_rna_ct_time.obs_names).melt(
	var_name="celltype",
	value_name="log_cpm"
)
#Plotting the distribution of values of the genes across all cell types in a density plot for RNA data
plt.figure(figsize=(10, 5))
sns.kdeplot(data=P_rna_scaled_plot, x="log_cpm", common_norm=False, fill=True, alpha=0.5)
plt.xlabel("log10((RNA + min(RNA)) * 1000000)")
plt.title("Density of log10((RNA + min(RNA)) * 1000000) across cell type + time")
plt.tight_layout()
plt.show()

In [ ]:
#taking the minimum non-zero value in the pb_atac_ct_time matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np

non_zero_mask = (pb_atac_ct_time.X > 0)
epsilon_atac = np.min(pb_atac_ct_time.X[non_zero_mask])

In [ ]:
#Plotting the distribution of the value of the genes across all cell types in a boxplot for RNA data
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

P_atac_scaled = np.log10((pb_atac_ct_time.X+ epsilon_atac)*10000000) 

# ATAC counts x samples
P_atac_scaled = P_atac_scaled.T.astype(float)

# convert to long format for seaborn
P_atac_scaled_plot = pd.DataFrame(P_atac_scaled, columns=pb_atac_ct_time.obs_names).melt(
	var_name="celltype",
	value_name="log_cpm"
)
#Plotting the distribution of values of the genes across all cell types in a density plot for ATAC data
plt.figure(figsize=(10, 5))
sns.kdeplot(data=P_atac_scaled_plot, x="log_cpm", common_norm=False, fill=True, alpha=0.5)
plt.xlabel("log10((ATAC + min(ATAC)) * 1000000)")
plt.title("Density of log10((ATAC + min(ATAC)) * 1000000) across cell type + time")
plt.tight_layout()
plt.show()

In [ ]:
pb_rna_ct_time.X = np.log10((pb_rna_ct_time.X+ epsilon_rna)*10000000)
pb_atac_ct_time.X = np.log10((pb_atac_ct_time.X+ epsilon_atac)*10000000)

# peak-wise correlation per gene
- For each gene, it loops through all assigned peaks. Computes one correlation per peak against the gene expression vector. Stores results as a nested dictionary:
- gene_peak_10kb_cor_results[gene_id][peak_id] = correlation
- This keeps full peak-level signal instead of collapsing peaks into one averaged accessibility profile.

In [11]:
#computing correlation for all positive windows by loading the function from the src folder 
from pathlib import Path
import sys

src_path = Path("/home/fgsasse_lrs_1/Downloads/BA/src/")

if str(src_path) not in sys.path:
       sys.path.insert(0, str(src_path)),

#import several functions from the src folder to classify peak-gene pairs and aggregate results across windows
from your_package.correlation import (
    add_correlation_categories,
    aggregate_correlation_categories,
    build_correlation_dataframe,
    compute_all_window_peak_correlations,
)

window_label = ["5kb","10kb", "20kb", "30kb", "40kb", "50kb", "60kb", "70kb", "80kb", "90kb", "100kb"]

window_assignments = { window: gene_peaks_pos[window] for window in window_label }
for window in window_assignments:
    print(f"Computing correlation for {window} window...")

all_window_results = compute_all_window_peak_correlations(
    atac_data=pb_atac_ct_time,
    rna_data=pb_rna_ct_time,
    window_assignments=window_assignments,
)

cor_res_df_pos = build_correlation_dataframe(
    all_window_results,
    window_labels=window_label,
)

#save the correlation results as a csv file
cor_res_df_pos.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Results/all_correlation_pos_df_ct_time_results.csv", index=False)



cor_res_df_pos = add_correlation_categories(cor_res_df_pos)
agg_cor_df_pos = aggregate_correlation_categories(cor_res_df_pos)

print(cor_res_df_pos.shape)
print(cor_res_df_pos.head())
print(agg_cor_df_pos.head())

Computing correlation for 5kb window...
Computing correlation for 10kb window...
Computing correlation for 20kb window...
Computing correlation for 30kb window...
Computing correlation for 40kb window...
Computing correlation for 50kb window...
Computing correlation for 60kb window...
Computing correlation for 70kb window...
Computing correlation for 80kb window...
Computing correlation for 90kb window...
Computing correlation for 100kb window...


/home/fgsasse_lrs_1/Downloads/BA/src/your_package/correlation.py:330: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["window", "gene", "category"])


(5032406, 9)
  window   gene           peak  correlation        pvalue  neglog10_pvalue  \
0    5kb  rpl24    1-9541-9969    -0.148999  3.713353e-02         1.430234   
1    5kb  cep97  1-13276-13705     0.285249  5.068191e-05         4.295147   
2    5kb  cep97  1-14059-14260     0.596123  3.032682e-20        19.518173   
3    5kb  cep97  1-14625-15105     0.702951  1.582080e-30        29.800772   
4    5kb  cep97  1-15724-15934     0.282058  6.197067e-05         4.207814   

           padj  neglog10_padj         category  
0  8.024557e-02       1.095579  non-significant  
1  2.583125e-04       3.587855    sig. positive  
2  1.807136e-18      17.743009    sig. positive  
3  3.388402e-28      27.470005    sig. positive  
4  3.095496e-04       3.509270    sig. positive  
  window  gene         category  count
0    5kb  a1cf    sig. negative      0
1    5kb  a1cf  non-significant      0
2    5kb  a1cf    sig. positive      2
3    5kb  a2ml    sig. negative      0
4    5kb  a2ml  non-sig

In [12]:
#computing correlation for all negative windows by loading the function from the src folder 
from pathlib import Path
import sys

src_path = Path("/home/fgsasse_lrs_1/Downloads/BA/src/")

if str(src_path) not in sys.path:
       sys.path.insert(0, str(src_path)),

#import several functions from the src folder to classify peak-gene pairs and aggregate results across windows
from your_package.correlation import (
    add_correlation_categories,
    aggregate_correlation_categories,
    build_correlation_dataframe,
    compute_all_window_peak_correlations,
)

window_label = ["5kb","10kb", "20kb", "30kb", "40kb", "50kb", "60kb", "70kb", "80kb", "90kb", "100kb"]
for window in window_assignments:
    print(f"Computing correlation for {window} window...")

window_assignments = { window: gene_peaks_neg[window] for window in window_label }
all_window_results = compute_all_window_peak_correlations(
    atac_data=pb_atac_ct_time,
    rna_data=pb_rna_ct_time,
    window_assignments=window_assignments,
)

cor_res_df_neg = build_correlation_dataframe(
    all_window_results,
    window_labels=window_label,
)

#save the correlation results as a csv file
cor_res_df_neg.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Results/all_correlation_neg_df_ct_time_results.csv", index=False)


cor_res_df_neg = add_correlation_categories(cor_res_df_neg)
agg_cor_df_neg = aggregate_correlation_categories(cor_res_df_neg)

print(cor_res_df_neg.shape)
print(cor_res_df_neg.head())
print(agg_cor_df_neg.head())

Computing correlation for 5kb window...
Computing correlation for 10kb window...
Computing correlation for 20kb window...
Computing correlation for 30kb window...
Computing correlation for 40kb window...
Computing correlation for 50kb window...
Computing correlation for 60kb window...
Computing correlation for 70kb window...
Computing correlation for 80kb window...
Computing correlation for 90kb window...
Computing correlation for 100kb window...


/home/fgsasse_lrs_1/Downloads/BA/src/your_package/correlation.py:330: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["window", "gene", "category"])


(5194274, 9)
  window   gene           peak  correlation        pvalue  neglog10_pvalue  \
0    5kb  rpl24  1-11007-12962    -0.304377  1.441907e-05         4.841063   
1    5kb  rpl24  1-13276-13705    -0.193392  6.610336e-03         2.179776   
2    5kb  rpl24  1-14059-14260    -0.484422  6.294299e-13        12.201053   
3    5kb  rpl24  1-14625-15105    -0.435176  1.841421e-10         9.734847   
4    5kb  rpl24  1-15724-15934    -0.247289  4.752497e-04         3.323078   

           padj  neglog10_padj       category  
0  6.728626e-05       4.172074  sig. negative  
1  1.672261e-02       1.776696  sig. negative  
2  9.140804e-12      11.039016  sig. negative  
3  1.892239e-09       8.723024  sig. negative  
4  1.613888e-03       2.792127  sig. negative  
  window  gene         category  count
0    5kb  a1cf    sig. negative      0
1    5kb  a1cf  non-significant      2
2    5kb  a1cf    sig. positive      1
3    5kb  a2ml    sig. negative      0
4    5kb  a2ml  non-significant    

In [13]:
#combine all the correlation dfs for both positive and negative windows into a single dataframe for downstream analysis and visualization
all_cor_combined_df = pd.concat(
    [cor_res_df_neg.assign(direction="neg"),
     cor_res_df_pos.assign(direction="pos")],
    ignore_index=True
)

#save the all_cor_dfs dataframe to a csv file for downstream analysis and visualization
all_cor_combined_df.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Results/all_correlation_combined_df_ct_time_results.csv", index=False)

In [14]:
#computing correlation for all combined windows by loading the function from the src folder 
from pathlib import Path
import sys

src_path = Path("/home/fgsasse_lrs_1/Downloads/BA/src/")

if str(src_path) not in sys.path:
       sys.path.insert(0, str(src_path)),

#import several functions from the src folder to classify peak-gene pairs and aggregate results across windows
from your_package.correlation import (
    add_correlation_categories,
    aggregate_correlation_categories,
    build_correlation_dataframe,
    compute_all_window_peak_correlations,
)

window_label = ["5kb","10kb", "20kb", "30kb", "40kb", "50kb", "60kb", "70kb", "80kb", "90kb", "100kb"]

window_assignments = { window: combined_gene_peaks[window] for window in window_label }
for window in window_assignments:
    print(f"Computing correlation for {window} window...")

all_window_results = compute_all_window_peak_correlations(
    atac_data=pb_atac_ct_time,
    rna_data=pb_rna_ct_time,
    window_assignments=window_assignments,
)

cor_res_range_df = build_correlation_dataframe(
    all_window_results,
    window_labels=window_label,
)

#save the correlation results as a csv file
cor_res_range_df.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Results/all_correlation_range_df_ct_time_results.csv", index=False)


cor_res_range_df = add_correlation_categories(cor_res_range_df)
agg_cor_range_df = aggregate_correlation_categories(cor_res_range_df)

print(cor_res_range_df.shape)
print(cor_res_range_df.head())
print(agg_cor_range_df.head())

Computing correlation for 5kb window...
Computing correlation for 10kb window...
Computing correlation for 20kb window...
Computing correlation for 30kb window...
Computing correlation for 40kb window...
Computing correlation for 50kb window...
Computing correlation for 60kb window...
Computing correlation for 70kb window...
Computing correlation for 80kb window...
Computing correlation for 90kb window...
Computing correlation for 100kb window...


/home/fgsasse_lrs_1/Downloads/BA/src/your_package/correlation.py:330: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["window", "gene", "category"])


(10226658, 9)
  window  gene                peak  correlation    pvalue  neglog10_pvalue  \
0    5kb  a1cf  12-6176521-6177141     0.315765  0.000007         5.184779   
1    5kb  a1cf  12-6174866-6175973     0.243407  0.000587         3.231693   
2    5kb  a1cf  12-6181654-6182495     0.125627  0.079348         1.100462   
3    5kb  a1cf  12-6180312-6181218     0.120367  0.092865         1.032148   
4    5kb  a1cf  12-6177594-6178380     0.263292  0.000193         3.715322   

       padj  neglog10_padj         category  
0  0.000035       4.452781    sig. positive  
1  0.002064       2.685358    sig. positive  
2  0.143603       0.842836  non-significant  
3  0.163485       0.786523  non-significant  
4  0.000762       3.118305    sig. positive  
  window  gene         category  count
0    5kb  a1cf    sig. negative      0
1    5kb  a1cf  non-significant      2
2    5kb  a1cf    sig. positive      3
3    5kb  a2ml    sig. negative      0
4    5kb  a2ml  non-significant      7


# Plotting the correlation results (ct_time)

CDF = Cumulative Distribution Function
- "What fraction of all peak–gene pairs in this window have padj ≤ t?

1) CDF curves

The curves shows the continuously across all thresholds:

- The 10kb curve rises the steepest near zero, meaning a large fraction of its pairs have very small padj values — its peaks are highly enriched for regulatory signal because they are all proximal to the TSS.
- Curves progressively flatten from 10kb → 100kb in the low-threshold region (left of the red dashed line), reflecting the dilution effect described above. The curves converge toward 1.0 at the right, which is expected — at padj = 1.0, all pairs are included by definition.
- The separation between curves is largest between 0 and ~0.3, exactly the biologically meaningful range. Beyond padj ~0.6 the curves bunch together, meaning the non-significant bulk of pairs behaves similarly regardless of window size.
Biological conclusion


The data shows proximity effect: regulatory peak–gene relationships are strongest and most enriched close to the TSS, with signal density falling off as the window is extend. The 10kb window is the most statistically clean. The additional ~188,000 significant pairs gained by going to 100kb represent candidate distal regulatory elements (enhancers) worth following up, but they come embedded in a much larger background of non-functional distal peaks and should be interpreted with higher scrutiny.

2) Proportion of significant peak-gene pairs
The bars show a clear monotonic decrease in the proportion of significant pairs as the window grows.The absolute number of significant pairs keeps growing (54k → 242k) even as the proportion shrinks. This tells us two things simultaneously:
- Larger windows do capture additional real regulatory peaks — it's not just adding noise, because the absolute count of significant pairs nearly quintuples from 10kb to 100kb
- But the majority of newly added distal peaks are non-regulatory — they dilute the signal, which is why BH correction becomes progressively stricter and the proportion falls

All four windows sit well above the 5% dashed baseline, which is the proportion we would expect under the null hypothesis of no signal. This confirms the correlation analysis is capturing genuine peak–gene associations and is not dominated by false discoveries.

Goal -> Best plot
- Overall shape comparison -> Overlapping KDE
- Median + spread in one view -> Violin + boxplot
- Rigorous, assumption-free comparison -> ECDF
- Simple biological summary -> Stacked bar

In [ ]:
#Plotting the proportion of significant correlations (padj < 0.05) for each window size
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# 1. Continuous proportion curve across thresholds 
thresholds   = np.linspace(0, 1, 500)          # padj thresholds from 0 to 1
window_labels = ["10kb", "20kb", "50kb", "100kb"]

# build a tidy table: one row per (window, threshold)
prop_records = []
for window_label in window_labels:
    subset = cor_res_df[cor_res_df["window"] == window_label]["padj"].dropna()
    n_total = len(subset)
    if n_total == 0:
        continue
    for t in thresholds:
        prop = (subset <= t).sum() / n_total
        prop_records.append({
            "window":    window_label,
            "threshold": t,
            "proportion": prop,
            "n_total":   n_total,
        })

prop_df = pd.DataFrame(prop_records)

# 2. Summary table at biologically meaningful cutoffs 
cutoffs = [0.001, 0.01, 0.05, 0.1, 0.2]
summary_records = []
for window_label in window_labels:
    subset = cor_res_df[cor_res_df["window"] == window_label]["padj"].dropna()
    n_total = len(subset)
    for t in cutoffs:
        n_sig  = (subset <= t).sum()
        prop   = n_sig / n_total if n_total > 0 else np.nan
        summary_records.append({
            "window":     window_label,
            "threshold":  t,
            "n_total":    n_total,
            "n_sig":      n_sig,
            "proportion": prop,
        })

summary_df = pd.DataFrame(summary_records)
print(summary_df.pivot(index="threshold", columns="window", values="proportion")
                .round(4)
                .to_string())

# 3. Plot 
palette = sns.color_palette("colorblind", n_colors=len(window_labels))
color_map = dict(zip(window_labels, palette))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.set_style("whitegrid")

# --- left panel: full proportion curve ---------------------------------------
ax = axes[0]
for window_label in window_labels:
    wdf = prop_df[prop_df["window"] == window_label]
    ax.plot(
        wdf["threshold"], wdf["proportion"],
        label=window_label,
        color=color_map[window_label],
        linewidth=2.0
    )

ax.axvline(0.05, color="crimson", linestyle="--",
           linewidth=1.2, alpha=0.85, label="padj = 0.05")
ax.set_xlabel("padj threshold", fontsize=11)
ax.set_ylabel("Proportion of pairs ≤ threshold", fontsize=11)
ax.set_title("Cumulative proportion of significant\npeak–gene pairs", fontsize=12)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(title="Window", fontsize=9, title_fontsize=9)

# --- right panel: bar chart at fixed cutoffs ---------------------------------
ax = axes[1]
bar_df = summary_df[summary_df["threshold"] == 0.05].copy()
bars = ax.bar(
    bar_df["window"],
    bar_df["proportion"],
    color=[color_map[w] for w in bar_df["window"]],
    edgecolor="white",
    linewidth=0.8,
    width=0.5
)

# annotate bars with absolute counts
for bar, (_, row) in zip(bars, bar_df.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"n={int(row['n_sig']):,}\n/{int(row['n_total']):,}",
        ha="center", va="bottom", fontsize=8.5
    )

ax.axhline(0.05, color="crimson", linestyle="--",
           linewidth=1.2, alpha=0.85, label="proportion = 0.05")
ax.set_xlabel("Window size", fontsize=11)
ax.set_ylabel("Proportion of pairs with padj ≤ 0.05", fontsize=11)
ax.set_title("Proportion significant at padj ≤ 0.05\nper window size", fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax.set_ylim(0, bar_df["proportion"].max() * 1.25)
ax.legend(fontsize=9)

fig.suptitle("Peak–gene pair significance across window sizes", y=1.02, fontsize=14)
plt.tight_layout()

fig.savefig(
    "/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Graphics/"
    "peak_gene_proportion_significant_ct_time.png",
    dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── classify every peak–gene pair into one of three categories ────────────────
def classify_pair(row):
    if row["padj"] <= 0.05 and row["correlation"] < 0:
        return "sig. negative"
    elif row["padj"] <= 0.05 and row["correlation"] > 0:
        return "sig. positive"
    else:
        return "non-significant"

cor_res_df["category"] = cor_res_df.apply(classify_pair, axis=1)

In [ ]:
# counting the number of significant peaks and non-significant peaks for each gene in all window sizes in cor_res_df, but with a category column instead of separate columns for sig pos, sig neg, and non-sig
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

agg_cor_df = (
    cor_res_df
    .dropna(subset=["correlation", "padj"])
    .groupby(["window", "gene", "category"])
    .size()
    .reset_index(name="count")
)
agg_cor_df.head()

# notice: the larger windows include the gene-peak pairs already occuring in the smaller window sizes!

In [ ]:
#Boxplot of correlation coefficients for significant pairs (padj ≤ 0.05) across window sizes
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# fixed order and colours for all panels
category_order  = ["sig. negative", "non-significant", "sig. positive"]
category_colors = {
    "sig. negative":  "#D32F2F",   # red
    "non-significant": "#90A4AE",  # grey
    "sig. positive":  "#388E3C",   # green
}

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 6), sharey=True)

for i, (ax, window_label) in enumerate(zip(axes, window_labels)):
    subset = cor_res_df[
        cor_res_df["window"] == window_label
    ].dropna(subset=["correlation", "padj"])

    if subset.empty:
        ax.text(0.5, 0.5, "No data",
                transform=ax.transAxes, ha="center", va="center")
        ax.set_title(f"Window = {window_label}", fontsize=12)
        continue

    # ── boxplot ───────────────────────────────────────────────────────────────
    sns.boxplot(
        data=subset,
        x="category",
        y="correlation",
        order=category_order,
        palette=category_colors,
        width=0.5,
        linewidth=1.2,
        flierprops=dict(
            marker=".",
            markersize=1.5,
            alpha=0.2,
            markeredgewidth=0
        ),
        ax=ax
    )

    # ── overlay stripplot for individual points (downsampled to avoid overplot)
    for cat in category_order:
        cat_data = subset[subset["category"] == cat]
        # downsample to max 500 points per category for readability
        if len(cat_data) > 1000:
            cat_data = cat_data.sample(1000, random_state=42)
        x_pos = category_order.index(cat)
        ax.scatter(
            np.random.normal(x_pos, 0.08, size=len(cat_data)),
            cat_data["correlation"],
            color=category_colors[cat],
            alpha=0.25,
            s=4,
            linewidths=0,
            zorder=2
        )

    # ── annotate with n per category ─────────────────────────────────────────
    for j, cat in enumerate(category_order):
        n = (subset["category"] == cat).sum()
        ax.text(
            j, ax.get_ylim()[0],
            f"n={n:,}",
            ha="center", va="bottom",
            fontsize=7.5, color="black"
        )

    # ── reference line at r = 0 ───────────────────────────────────────────────
    ax.axhline(0.0, color="black", linestyle="--",
               linewidth=1.0, alpha=0.6)

    ax.set_title(f"Window = {window_label}", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("Pearson r" if i == 0 else "", fontsize=11)
    ax.set_xticklabels(category_order, fontsize=9, rotation=15, ha="right")

# ── shared legend ─────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=category_colors[cat], label=cat)
    for cat in category_order
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=3,
    fontsize=9,
    frameon=True,
    bbox_to_anchor=(0.5, -0.04)
)

fig.suptitle(
    "Pearson r distribution by significance category across window sizes",
    y=1.02, fontsize=14
)
plt.tight_layout()

fig.savefig(
    "/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Graphics/"
    "peak_gene_correlation_boxplot_categories_ct_time.png",
    dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
# Plotting the distribution of the number of peaks per gene across all window sizes
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

# ensure plot order is stable
window_labels = ["10kb", "20kb", "50kb", "100kb"]
category_order = ["sig. negative", "non-significant", "sig. positive"]
category_colors = {
    "sig. negative": "#D32F2F",
    "non-significant": "#90A4AE",
    "sig. positive": "#388E3C",
}

# agg_df is already in long format: (gene, window, category, count)
# ensure window is categorical for proper ordering
plot_df = agg_cor_df.copy()
plot_df["window"] = pd.Categorical(
    plot_df["window"],
    categories=window_labels,
    ordered=True,
)

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

sns.boxplot(
    data=plot_df.dropna(subset=["count"]),
    x="window",
    y="count",
    hue="category",
    hue_order=category_order,
    order=window_labels,
    palette=category_colors,
    width=0.6,
    linewidth=1.2,
    flierprops=dict(
        marker=".",
        markersize=1.5,
        alpha=0.2,
        markeredgewidth=0,
    ),
    ax=ax,
)

# annotate median count for each window-category box
n_groups = len(category_order)
total_width = 0.6
box_width = total_width / n_groups

for w_idx, window_label in enumerate(window_labels):
    w_subset = plot_df[plot_df["window"] == window_label]
    for c_idx, cat in enumerate(category_order):
        vals = w_subset[w_subset["category"] == cat]["count"]
        if vals.empty:
            continue

        x_pos = (
            w_idx
            - total_width / 2
            + box_width / 2
            + c_idx * box_width
        )
        y_med = vals.median()
        y_mean = vals.mean()
        ax.text(
            x=x_pos,
            y=ax.get_ylim()[0] + 0.00001  * (ax.get_ylim()[1] - ax.get_ylim()[0]), # place text slightly above x-axis
            s=f"median ={int(y_med):,}\nmean ={int(y_mean):,}",
            ha="center",
            va="bottom",
            fontsize=6,
            fontweight="bold",
            color=category_colors[cat],
        )

# ── labels + legend ───────────────────────────────────────────────────────────
ax.set_xlabel("Window size", fontsize=12)
ax.set_ylabel("Number of peaks per gene", fontsize=12)
ax.set_title("Distribution of peak counts per gene across window sizes (grouped by correlation category)", fontsize=13)

legend_patches = [
    mpatches.Patch(color=category_colors[cat], label=cat)
    for cat in category_order
]
ax.legend(
    handles=legend_patches,
    title="Correlation Category\n(based on padj ≤ 0.05)",
    fontsize=9,
    title_fontsize=9,
    frameon=True,
    loc="upper left",
)

plt.tight_layout()

fig.savefig(
    "/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Graphics/"
    "peak_gene_count_boxplot_aggregated_ct_time.png",
    dpi=300, bbox_inches="tight"
)
plt.show()